## HTML files analysis

### Load

In [5]:
import pandas as pd
import numpy as np
import ast

CSV_PATH = "../data/evaluation/metadata_html_samples_cleaned.csv"

df = pd.read_csv(CSV_PATH, dtype={"internal_id": "string"})

# Converti query_params da string a dict
def safe_eval(x):
    try:
        return ast.literal_eval(x)
    except:
        return {}

df["query_params"] = df["query_params"].apply(safe_eval)

# === CONVERSIONE SIZE IN KB ===
df["size_kb"] = df["size"] / 1024

### STATISTICHE GENERALI

In [6]:
print(" - Numero totale documenti:", len(df))
print(" -  Numero domini unici:", df["domain"].nunique())
print(" -  Numero internal_id unici:", df["internal_id"].nunique())
print(" -  Numero base_path unici:", df["base_path"].nunique())

 - Numero totale documenti: 16453
 -  Numero domini unici: 4
 -  Numero internal_id unici: 165
 -  Numero base_path unici: 262


### DOCUMENTI PER DOMINIO

In [7]:
print("\n==============================")
print(" DOCUMENTI PER DOMINIO")
print("==============================\n")

print(df["domain"].value_counts().head(10))


 DOCUMENTI PER DOMINIO

domain
docenti.unisa.it          9581
www.diem.unisa.it         6620
corsi.unisa.it             251
www.diem.unisa.it.html       1
Name: count, dtype: int64


# DISTRIBUZIONE DEPTH

In [8]:
print("\n==============================")
print(" DISTRIBUZIONE DEPTH")
print("==============================\n")

print(df["depth"].value_counts().sort_index())


 DISTRIBUZIONE DEPTH

depth
0     174
1     505
2    2291
3    3878
4    2985
5    6620
Name: count, dtype: int64


### ID (Matricole)

In [9]:
print("\n==============================")
print(" TOP INTERNAL_ID")
print("==============================\n")

print(df["internal_id"].value_counts().head(10))


 TOP INTERNAL_ID

internal_id
001295    464
005768    373
005501    344
003741    336
023586    271
005630    261
004491    253
001366    242
023604    240
004687    233
Name: count, dtype: int64[pyarrow]


### ANALISI DIMENSIONI FILE

In [10]:
print("\n==============================")
print(" ANALISI DIMENSIONI FILE (KB)")
print("==============================\n")

percentiles = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]

size_stats = df["size_kb"].describe(percentiles=percentiles)
print(size_stats)

print("\n- Media (KB):", round(df["size_kb"].mean(), 2))
print("- Mediana (KB):", round(df["size_kb"].median(), 2))
print("- Std Dev (KB):", round(df["size_kb"].std(), 2))



 ANALISI DIMENSIONI FILE (KB)

count    16453.000000
mean         5.737272
std         16.073548
min          0.179688
1%           0.253906
5%           0.757812
10%          1.159180
25%          1.605469
50%          2.383789
75%          7.317383
90%         12.604492
95%         15.197266
99%         34.654727
max        675.576172
Name: size_kb, dtype: float64

- Media (KB): 5.74
- Mediana (KB): 2.38
- Std Dev (KB): 16.07


In [11]:
print("\n==============================")
print(" DISTRIBUZIONE DIMENSIONI (BINNING)")
print("==============================\n")

bins = [0, 1, 2, 3, 4, 5, 10, 20, 50, 100, np.inf]
labels = [
    "0-1 KB", "1-2 KB", "2-3 KB", "3-4 KB", "4-5 KB",
    "5-10 KB", "10-20 KB", "20-50 KB", "50-100 KB", ">100 KB"
]

df["size_bin"] = pd.cut(df["size_kb"], bins=bins, labels=labels)

bin_counts = df["size_bin"].value_counts().sort_index()
bin_percent = df["size_bin"].value_counts(normalize=True).sort_index() * 100

print("Conteggi:")
print(bin_counts)

print("\nPercentuali:")
print(bin_percent.round(2).astype(str) + " %")


 DISTRIBUZIONE DIMENSIONI (BINNING)

Conteggi:
size_bin
0-1 KB       1255
1-2 KB       5342
2-3 KB       3032
3-4 KB       1223
4-5 KB        528
5-10 KB      2245
10-20 KB     2422
20-50 KB      322
50-100 KB      44
>100 KB        40
Name: count, dtype: int64

Percentuali:
size_bin
0-1 KB        7.63 %
1-2 KB       32.47 %
2-3 KB       18.43 %
3-4 KB        7.43 %
4-5 KB        3.21 %
5-10 KB      13.64 %
10-20 KB     14.72 %
20-50 KB      1.96 %
50-100 KB     0.27 %
>100 KB       0.24 %
Name: proportion, dtype: str


### OUTLIER 

In [12]:
print("\n==============================")
print(" FILE PIÙ GRANDI")
print("==============================\n")

largest = df.sort_values(by="size_kb", ascending=False).head(10)
print(largest[["doc_id", "domain", "size_kb"]])


 FILE PIÙ GRANDI

       doc_id             domain     size_kb
16452    2858  www.diem.unisa.it  675.576172
16451   28163  www.diem.unisa.it  672.244141
16450   27410  www.diem.unisa.it  603.464844
16449   13096  www.diem.unisa.it  536.764648
16448   28428  www.diem.unisa.it  470.337891
16447   45811  www.diem.unisa.it  452.697266
16446   27405  www.diem.unisa.it  432.259766
16445   12792  www.diem.unisa.it  409.322266
16444   28427  www.diem.unisa.it  375.276367
16443   45703  www.diem.unisa.it  358.562500


### QUERY PARAMS

In [13]:
print("\n==============================")
print(" ANALISI QUERY PARAMS")
print("==============================\n")

param_counts = {}

for params in df["query_params"]:
    for key in params:
        param_counts[key] = param_counts.get(key, 0) + 1

sorted_params = sorted(param_counts.items(), key=lambda x: x[1], reverse=True)

for k, v in sorted_params[:10]:
    print(f"{k}: {v}")


 ANALISI QUERY PARAMS

anno: 10189
modulo: 5766
bando: 5555
id: 3474
cId: 2500
pId: 2500
ruolo: 2187
progetto: 2023
stato: 1924
tip: 1737


### CROSS ANALYSIS

In [14]:
print("\n==============================")
print(" CROSS ANALYSIS (domain vs depth)")
print("==============================\n")

pivot = pd.pivot_table(
    df,
    index="domain",
    columns="depth",
    values="doc_id",
    aggfunc="count",
    fill_value=0
)
print(pivot.head(10))


 CROSS ANALYSIS (domain vs depth)

depth                     0    1     2     3     4     5
domain                                                  
corsi.unisa.it            7  115    90    25     5     9
docenti.unisa.it        166  378  2133  3015  1958  1931
www.diem.unisa.it         0   12    68   838  1022  4680
www.diem.unisa.it.html    1    0     0     0     0     0


In [15]:
print("\n==============================")
print(" INSIGHT ")
print("==============================\n")

print("Distribuzione depth (normalizzata):")
print(df["depth"].value_counts(normalize=True).sort_index())

print("\nTop domini (%):")
print(df["domain"].value_counts(normalize=True).head(5))


 INSIGHT 

Distribuzione depth (normalizzata):
depth
0    0.010576
1    0.030693
2    0.139245
3    0.235702
4    0.181426
5    0.402358
Name: proportion, dtype: float64

Top domini (%):
domain
docenti.unisa.it          0.582325
www.diem.unisa.it         0.402358
corsi.unisa.it            0.015256
www.diem.unisa.it.html    0.000061
Name: proportion, dtype: float64
